# Real-ESRGAN Upscaler Web UI for Google Colab

โน้ตบุ๊กนี้สร้างเว็บอัปสเกลภาพ/วิดีโอด้วย Real-ESRGAN บน Google Colab และเปิด URL สาธารณะด้วย Cloudflare Quick Tunnel โดยไม่มีโฆษณาในหน้าเว็บของเรา

## วิธีใช้
1. เปิด Runtime เป็น GPU: `Runtime > Change runtime type > T4 GPU` หรือ GPU อื่น
2. รันทุกเซลล์จากบนลงล่าง
3. เปิดลิงก์ `trycloudflare.com` ที่แสดงท้ายเซลล์สุดท้าย
4. อัปโหลดรูปหรือวิดีโอ เลือกโมเดล Anime/Normal แล้วกด Upscale


In [ ]:
#@title 1) Install Real-ESRGAN, Gradio, FFmpeg helpers, and Cloudflare Tunnel
import os, pathlib, subprocess, sys, textwrap

def run(cmd, cwd=None):
    print(f'\n$ {cmd}')
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

REPO_DIR = pathlib.Path('/content/Real-ESRGAN')

run('apt-get -y update >/dev/null && apt-get -y install ffmpeg wget curl >/dev/null')
run(f'{sys.executable} -m pip install -U pip wheel setuptools >/dev/null')
run(f'{sys.executable} -m pip install -U gradio tqdm opencv-python-headless pillow numpy >/dev/null')

if not REPO_DIR.exists():
    run('git clone --depth 1 https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN')

run(f'{sys.executable} -m pip install -r requirements.txt >/dev/null', cwd=str(REPO_DIR))
run(f'{sys.executable} -m pip install -e . >/dev/null', cwd=str(REPO_DIR))

# Compatibility patch for newer torchvision versions where functional_tensor was moved.
import site
for base in site.getsitepackages() + [site.getusersitepackages()]:
    p = pathlib.Path(base) / 'basicsr/data/degradations.py'
    if not p.exists():
        continue
    s = p.read_text()
    patched = s.replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                        'from torchvision.transforms.functional import rgb_to_grayscale')
    if patched != s:
        p.write_text(patched)
        print('Patched basicsr torchvision compatibility:', p)

CLOUDFLARED = pathlib.Path('/usr/local/bin/cloudflared')
if not CLOUDFLARED.exists():
    run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared')
    run('chmod +x /usr/local/bin/cloudflared')

print('✅ Install complete. Runtime:', sys.version)


In [ ]:
#@title 2) Build processing functions for images and videos
import json, math, os, pathlib, re, shutil, subprocess, sys, time, uuid
from typing import Optional, Tuple

import gradio as gr

REPO_DIR = pathlib.Path('/content/Real-ESRGAN')
WORK_DIR = pathlib.Path('/content/realesrgan_jobs')
WORK_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CHOICES = {
    'Normal • RealESRGAN x4+ (sharp photos)': {'model': 'RealESRGAN_x4plus', 'scale': 4, 'anime': False},
    'Normal • RealESRNet x4+ (softer photos)': {'model': 'RealESRNet_x4plus', 'scale': 4, 'anime': False},
    'Normal • RealESRGAN x2+ (faster / lighter)': {'model': 'RealESRGAN_x2plus', 'scale': 2, 'anime': False},
    'Anime • RealESRGAN x4+ Anime 6B': {'model': 'RealESRGAN_x4plus_anime_6B', 'scale': 4, 'anime': True},
    'Anime Video • realesr-animevideov3': {'model': 'realesr-animevideov3', 'scale': 4, 'anime': True},
}
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
VIDEO_EXTS = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}

def safe_name(name: str) -> str:
    stem = pathlib.Path(name).stem or 'upload'
    stem = re.sub(r'[^A-Za-z0-9._-]+', '_', stem).strip('._') or 'upload'
    return stem[:80]

def run_cmd(cmd, progress: Optional[gr.Progress] = None, label: str = ''):
    if label and progress:
        progress(0, desc=label)
    print(' '.join(map(str, cmd)))
    completed = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if completed.returncode != 0:
        raise gr.Error(completed.stdout[-4000:])
    return completed.stdout

def probe_video(path: pathlib.Path) -> Tuple[float, float]:
    cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=r_frame_rate,duration', '-of', 'json', str(path)]
    data = json.loads(subprocess.check_output(cmd, text=True))
    stream = data['streams'][0]
    fps_text = stream.get('r_frame_rate', '30/1')
    num, den = [float(x) for x in fps_text.split('/')]
    fps = num / den if den else 30.0
    duration = float(stream.get('duration') or 0.0)
    return fps, duration

def upscale_image(input_path: pathlib.Path, model_label: str, outscale: int, face_enhance: bool, tile: int, progress: gr.Progress) -> pathlib.Path:
    cfg = MODEL_CHOICES[model_label]
    job = WORK_DIR / f'image_{uuid.uuid4().hex[:8]}'
    out_dir = job / 'out'
    out_dir.mkdir(parents=True, exist_ok=True)
    progress(0.1, desc='Preparing image')
    cmd = [sys.executable, 'inference_realesrgan.py', '-n', cfg['model'], '-i', str(input_path), '-o', str(out_dir), '--outscale', str(outscale), '--tile', str(tile)]
    if face_enhance and not cfg['anime']:
        cmd.append('--face_enhance')
    run_cmd(cmd, progress, 'Upscaling image with Real-ESRGAN')
    progress(0.92, desc='Preparing download')
    outputs = sorted(out_dir.glob('*'))
    if not outputs:
        raise gr.Error('No output image was created.')
    final = job / f'{safe_name(input_path.name)}_{cfg["model"]}_x{outscale}.png'
    shutil.move(str(outputs[0]), final)
    progress(1, desc='Done')
    return final

def upscale_video(input_path: pathlib.Path, model_label: str, outscale: int, face_enhance: bool, tile: int, progress: gr.Progress) -> pathlib.Path:
    cfg = MODEL_CHOICES[model_label]
    job = WORK_DIR / f'video_{uuid.uuid4().hex[:8]}'
    frames = job / 'frames'
    upscaled = job / 'upscaled'
    frames.mkdir(parents=True, exist_ok=True)
    upscaled.mkdir(parents=True, exist_ok=True)
    fps, duration = probe_video(input_path)
    progress(0.05, desc='Extracting video frames')
    run_cmd(['ffmpeg', '-y', '-i', str(input_path), '-vsync', '0', str(frames / 'frame_%08d.png')])
    frame_files = sorted(frames.glob('*.png'))
    if not frame_files:
        raise gr.Error('No frames were extracted from the video.')

    # Process frames in small batches so the Gradio progress bar keeps moving on long videos.
    batch_size = 12 if cfg['anime'] else 8
    total_batches = math.ceil(len(frame_files) / batch_size)
    for batch_index, start in enumerate(range(0, len(frame_files), batch_size), start=1):
        batch_files = frame_files[start:start + batch_size]
        batch_in = job / f'batch_in_{batch_index:04d}'
        batch_out = job / f'batch_out_{batch_index:04d}'
        batch_in.mkdir(parents=True, exist_ok=True)
        batch_out.mkdir(parents=True, exist_ok=True)
        for frame in batch_files:
            shutil.copy2(frame, batch_in / frame.name)
        progress(0.18 + 0.58 * ((batch_index - 1) / total_batches), desc=f'Upscaling frame batch {batch_index}/{total_batches}')
        cmd = [sys.executable, 'inference_realesrgan.py', '-n', cfg['model'], '-i', str(batch_in), '-o', str(batch_out), '--outscale', str(outscale), '--tile', str(tile)]
        if face_enhance and not cfg['anime']:
            cmd.append('--face_enhance')
        run_cmd(cmd)
        for produced in batch_out.glob('*'):
            target_name = produced.name.replace('_out', '') if produced.name.endswith('_out.png') else produced.name
            if not target_name.endswith('.png'):
                target_name = pathlib.Path(target_name).with_suffix('.png').name
            shutil.move(str(produced), upscaled / target_name)
        shutil.rmtree(batch_in, ignore_errors=True)
        shutil.rmtree(batch_out, ignore_errors=True)

    progress(0.82, desc='Rebuilding video')
    final = job / f'{safe_name(input_path.name)}_{cfg["model"]}_x{outscale}.mp4'
    frame_pattern = str(upscaled / 'frame_%08d.png')
    silent_video = job / 'silent.mp4'
    run_cmd(['ffmpeg', '-y', '-framerate', f'{fps:.6f}', '-i', frame_pattern, '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(silent_video)])

    progress(0.92, desc='Adding original audio if available')
    mux = subprocess.run(['ffmpeg', '-y', '-i', str(silent_video), '-i', str(input_path), '-map', '0:v:0', '-map', '1:a?', '-c:v', 'copy', '-c:a', 'aac', '-shortest', str(final)], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if mux.returncode != 0:
        shutil.copy2(silent_video, final)
    progress(1, desc='Done')
    return final

def process_upload(uploaded_file, model_label, outscale, face_enhance, tile, progress=gr.Progress(track_tqdm=True)):
    if uploaded_file is None:
        raise gr.Error('Please upload an image or video first.')
    src = pathlib.Path(uploaded_file.name)
    ext = src.suffix.lower()
    progress(0.02, desc='Starting')
    if ext in IMAGE_EXTS:
        out = upscale_image(src, model_label, int(outscale), bool(face_enhance), int(tile), progress)
        return str(out), gr.update(value=str(out), visible=True), gr.update(value=None, visible=False), f'✅ เสร็จแล้ว: {out.name}'
    if ext in VIDEO_EXTS:
        out = upscale_video(src, model_label, int(outscale), bool(face_enhance), int(tile), progress)
        return str(out), gr.update(value=None, visible=False), gr.update(value=str(out), visible=True), f'✅ เสร็จแล้ว: {out.name}'
    raise gr.Error('รองรับเฉพาะไฟล์ภาพ PNG/JPG/WEBP/BMP และวิดีโอ MP4/MOV/MKV/WEBM/AVI')


In [ ]:
#@title 3) Launch the no-ad web UI and expose it with Cloudflare Quick Tunnel
import os, pathlib, re, socket, subprocess, threading, time, urllib.error, urllib.parse, urllib.request
from IPython.display import HTML, display
import gradio as gr

PORT = 7860

CSS = r'''
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800;900&family=Noto+Sans+Thai:wght@400;600;800&display=swap');
:root { --glow-a:#8b5cf6; --glow-b:#06b6d4; --glow-c:#f97316; }
body, .gradio-container {
  font-family: 'Inter','Noto Sans Thai',sans-serif !important;
  background: radial-gradient(circle at 15% 10%, rgba(139,92,246,.35), transparent 34%),
              radial-gradient(circle at 85% 0%, rgba(6,182,212,.28), transparent 30%),
              radial-gradient(circle at 55% 105%, rgba(249,115,22,.20), transparent 35%),
              #060712 !important;
  color: #f8fafc !important;
}
.gradio-container { max-width: 1180px !important; margin: auto !important; }
.hero {
  border: 1px solid rgba(255,255,255,.16); border-radius: 32px; padding: 34px; margin: 18px 0;
  background: linear-gradient(135deg, rgba(255,255,255,.16), rgba(255,255,255,.04));
  box-shadow: 0 30px 90px rgba(0,0,0,.35), inset 0 1px 0 rgba(255,255,255,.24);
  backdrop-filter: blur(22px); position: relative; overflow: hidden;
}
.hero:after { content:''; position:absolute; inset:-80px; background: conic-gradient(from 120deg, transparent, rgba(139,92,246,.18), transparent, rgba(6,182,212,.16), transparent); animation: spin 12s linear infinite; pointer-events:none; }
.hero > * { position: relative; z-index: 1; }
.eyebrow { letter-spacing:.24em; text-transform:uppercase; color:#a5b4fc; font-weight:800; font-size:12px; }
.title { font-size: clamp(36px, 7vw, 86px); line-height:.92; font-weight:900; margin: 12px 0; }
.title span { background: linear-gradient(90deg,#fff,#67e8f9,#c4b5fd,#fed7aa); -webkit-background-clip:text; color:transparent; }
.subtitle { color:#cbd5e1; font-size:18px; max-width:820px; }
.glass, .gradio-container .form, .gradio-container .panel { border-radius: 24px !important; border:1px solid rgba(255,255,255,.14) !important; background: rgba(15,23,42,.64) !important; box-shadow: 0 22px 60px rgba(0,0,0,.28) !important; }
button.primary, .gr-button-primary { border:0 !important; border-radius: 999px !important; font-weight:900 !important; background: linear-gradient(90deg,var(--glow-a),var(--glow-b)) !important; box-shadow: 0 14px 40px rgba(6,182,212,.25) !important; }
.footer-note { text-align:center; color:#94a3b8; font-size:13px; margin: 18px 0 4px; }
@keyframes spin { to { transform: rotate(1turn); } }
'''

with gr.Blocks(css=CSS, title='Nebula Real-ESRGAN Upscaler', theme=gr.themes.Soft(primary_hue='violet', secondary_hue='cyan')) as demo:
    gr.HTML('''
    <section class="hero">
      <div class="eyebrow">Cloudflare Quick Tunnel • Real-ESRGAN • No Ads</div>
      <div class="title">Nebula <span>Upscaler</span></div>
      <p class="subtitle">อัปโหลดภาพหรือวิดีโอ เลือกโมเดล Anime/Normal ดู progress ขณะประมวลผล แล้วดาวน์โหลดไฟล์พร้อม preview ได้ทันที</p>
    </section>
    ''')
    with gr.Row(equal_height=True):
        with gr.Column(scale=5, elem_classes=['glass']):
            upload_box = gr.File(label='อัปโหลดไฟล์ภาพหรือวิดีโอ', file_types=['image', 'video'])
            model = gr.Radio(choices=list(MODEL_CHOICES.keys()), value='Normal • RealESRGAN x4+ (sharp photos)', label='เลือกโมเดล Real-ESRGAN')
            with gr.Row():
                outscale = gr.Slider(1, 4, value=4, step=1, label='Output scale')
                tile = gr.Slider(0, 512, value=0, step=64, label='Tile size (0 = auto / best quality)')
            face = gr.Checkbox(value=False, label='Face enhance สำหรับภาพคน (ใช้กับโมเดล Normal เท่านั้น)')
            run_btn = gr.Button('✨ Upscale Now', variant='primary')
        with gr.Column(scale=6, elem_classes=['glass']):
            status = gr.Markdown('พร้อมเริ่มงาน — อัปโหลดไฟล์แล้วกดปุ่มได้เลย')
            image_preview = gr.Image(label='Image preview', visible=True, height=420)
            video_preview = gr.Video(label='Video preview', visible=False, height=420)
            download = gr.File(label='ดาวน์โหลดไฟล์ผลลัพธ์')
    gr.HTML('<div class="footer-note">Tip: วิดีโอยาวมากจะใช้เวลานาน ควรทดสอบด้วยคลิปสั้นก่อน • หน้านี้ไม่มีโฆษณา</div>')

    run_btn.click(process_upload, inputs=[upload_box, model, outscale, face, tile], outputs=[download, image_preview, video_preview, status])

def launch_gradio_once():
    """Start Gradio once, then wait until the local HTTP server really responds."""
    global gradio_thread
    if 'gradio_thread' not in globals() or not gradio_thread.is_alive():
        gradio_thread = threading.Thread(
            target=lambda: demo.queue(default_concurrency_limit=1).launch(
                server_name='0.0.0.0',
                server_port=PORT,
                share=False,
                quiet=True,
                prevent_thread_lock=True,
            ),
            daemon=True,
        )
        gradio_thread.start()

    local_url = f'http://127.0.0.1:{PORT}/'
    for _ in range(90):
        try:
            with urllib.request.urlopen(local_url, timeout=2) as response:
                if response.status < 500:
                    print('✅ Gradio พร้อมใช้งานใน Colab:', local_url)
                    return
        except Exception:
            time.sleep(1)
    raise RuntimeError('Gradio local server did not become ready. กรุณารันเซลล์นี้ใหม่อีกครั้ง')

def stop_previous_cloudflared():
    global cf
    if 'cf' in globals() and cf.poll() is None:
        cf.terminate()
        try:
            cf.wait(timeout=5)
        except subprocess.TimeoutExpired:
            cf.kill()

def dns_ready(url: str) -> bool:
    host = urllib.parse.urlparse(url).hostname
    if not host:
        return False
    try:
        socket.getaddrinfo(host, 443)
        return True
    except socket.gaierror:
        return False

def http_ready(url: str) -> bool:
    try:
        request = urllib.request.Request(url, headers={'User-Agent': 'Colab tunnel readiness check'})
        with urllib.request.urlopen(request, timeout=8) as response:
            return response.status < 500
    except urllib.error.HTTPError as exc:
        return exc.code < 500
    except Exception:
        return False

def start_cloudflare_tunnel(max_attempts=3):
    """
    Cloudflare sometimes prints a trycloudflare.com URL before DNS has propagated.
    To avoid DNS_PROBE_FINISHED_NXDOMAIN, only show the URL after DNS and HTTP checks pass.
    """
    global cf
    for attempt in range(1, max_attempts + 1):
        stop_previous_cloudflared()
        cloudflared_log = pathlib.Path(f'/content/cloudflared_attempt_{attempt}.log')
        cloudflared_log.write_text('')
        print(f'🚇 เริ่ม Cloudflare Quick Tunnel ครั้งที่ {attempt}/{max_attempts} ...')
        cf = subprocess.Popen(
            ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate', '--loglevel', 'info'],
            stdout=open(cloudflared_log, 'a'),
            stderr=subprocess.STDOUT,
            text=True,
        )

        public_url = None
        for _ in range(60):
            if cf.poll() is not None:
                break
            text = cloudflared_log.read_text(errors='ignore')
            match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', text)
            if match:
                public_url = match.group(0)
                print('🔗 ได้ URL แล้ว กำลังรอ DNS/เว็บพร้อมใช้งาน:', public_url)
                break
            time.sleep(1)

        if not public_url:
            print('⚠️ ยังไม่ได้ URL จาก cloudflared รอบนี้')
            print(cloudflared_log.read_text(errors='ignore')[-2000:])
            continue

        for second in range(1, 121):
            if cf.poll() is not None:
                print('⚠️ cloudflared หยุดทำงานก่อนเว็บพร้อม จะลองสร้าง tunnel ใหม่')
                break
            if dns_ready(public_url) and http_ready(public_url):
                print('✅ Cloudflare Tunnel พร้อมใช้งานจริง:', public_url)
                display(HTML(f'''
                <div style="padding:18px;border-radius:18px;background:#0f172a;color:#fff;font-family:sans-serif">
                  <div style="font-size:15px;color:#93c5fd;margin-bottom:8px">Nebula Upscaler is online</div>
                  <a href="{public_url}" target="_blank" style="font-size:22px;color:#67e8f9;font-weight:800">เปิดเว็บ: {public_url}</a>
                  <div style="margin-top:10px;color:#cbd5e1">ถ้าเพิ่งกดแล้วเข้าไม่ได้ ให้รอ 10-20 วินาทีแล้ว refresh แต่ URL นี้ผ่าน DNS/HTTP check แล้ว</div>
                </div>
                '''))
                return public_url
            if second % 10 == 0:
                print(f'⏳ รอ DNS/HTTP ของ tunnel อีกสักครู่... {second}s')
            time.sleep(1)

    raise RuntimeError('สร้าง Cloudflare Tunnel ไม่สำเร็จ หรือ DNS ยังไม่พร้อมหลังลองหลายครั้ง กรุณารันเซลล์นี้ใหม่')

launch_gradio_once()
public_url = start_cloudflare_tunnel()
